# 🚕 DAS7002 – Big Data Technologies
## NYC Yellow Taxi Trip Data Analysis Using Apache Spark and MLlib

| | |
|---|---|
| **Student ID** | 20318851 |
| **Module** | DAS7002 – Big Data Technologies |
| **Assessment** | PRAC1 – 75% Weighting |
| **Institution** | Cardiff Metropolitan University |
| **Academic Year** | 2025–26, Semester 2 |

---

### 📋 About This Notebook

This notebook analyses three months of NYC Yellow Taxi trip records (July–September 2025) using Apache PySpark in a distributed computing environment. It covers four interconnected tasks:

1. **Task 1 – Data Preparation:** Loading, cleaning, and engineering the dataset
2. **Task 2 – EDA:** Eight exploratory questions across temporal, spatial, payment, and behavioural dimensions
3. **Task 3 – Clustering:** KMeans-based trip segmentation using MLlib
4. **Task 4 – Supervised Learning:** Demand classification using Decision Trees and Logistic Regression

---

### 📂 Before Running
Upload your three Parquet files using the **Files panel (📁) on the left sidebar**:
- `yellow_tripdata_2025-07.parquet`
- `yellow_tripdata_2025-08.parquet`
- `yellow_tripdata_2025-09.parquet`

Then run all cells from top to bottom using **Runtime → Run all**.

---
## ⚙️ Environment Setup – Installing PySpark

In [ ]:
# Student ID: 20318851
# First things first — we need to install PySpark and findspark into this Colab environment.
# PySpark is not available by default in Colab, so we install it fresh each session.
# findspark helps Python locate the Spark installation automatically.

!pip install pyspark --quiet
!pip install findspark --quiet

print('✅ PySpark and findspark installed successfully.')
print('   You can now initialise a Spark session in the next cell.')

> **Output explained:** This cell simply confirms that PySpark has been installed into the Colab runtime. If you see any red warnings about package conflicts, they can safely be ignored — PySpark will still work correctly.

In [ ]:
# Set the path to the Parquet files.
# The wildcard (*) pattern tells Spark to load all three monthly files at once
# rather than reading them one by one — much more efficient!

# If your files are in Google Drive instead, comment out the line below
# and uncomment the Drive section:
# from google.colab import drive
# drive.mount('/content/drive')
# PARQUET_PATH = '/content/drive/MyDrive/NYC_Taxi/yellow_tripdata_2025-*.parquet'

PARQUET_PATH = '/content/yellow_tripdata_2025-*.parquet'
STUDENT_ID   = '20318851'

print(f'Student ID   : {STUDENT_ID}')
print(f'Parquet path : {PARQUET_PATH}')
print('Ready to initialise Spark!')

> **Output explained:** Confirms the file path Spark will use to locate the Parquet data. The wildcard `*` means all three monthly files will be loaded in one go.

---
# 📦 Task 1 – Data Preparation (15%)

Before we can analyse anything, we need to get the data into a clean, consistent shape. Raw taxi data from real-world sources almost always contains missing values, impossible entries, and redundant columns. This task takes care of all of that — loading the raw Parquet files, stripping out what we don't need, handling missing values sensibly, removing statistical outliers, and engineering some extra columns (like trip duration and pickup hour) that will prove useful throughout the analysis.

In [ ]:
# ── 1.0 Initialise the Spark Session ───────────────────────────────────────
# The SparkSession is our entry point to everything Spark.
# We configure it with enough memory to handle tens of millions of rows
# and enable Adaptive Query Execution (AQE), which lets Spark dynamically
# re-optimise shuffle operations at runtime.

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when
from pyspark import StorageLevel

spark = SparkSession.builder \
    .appName(f'NYC_Taxi_DAS7002_{STUDENT_ID}') \
    .config('spark.sql.shuffle.partitions', '100') \
    .config('spark.driver.memory', '4g') \
    .config('spark.executor.memory', '4g') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'✅ Spark {spark.version} started for student {STUDENT_ID}')
print(f'   App name : {spark.sparkContext.appName}')
print(f'   Master   : {spark.sparkContext.master}')

> **Output explained:** Spark is now running as a local cluster inside Colab. The app name includes your student ID (20318851) to personalise the session. The master `local[*]` means Spark is using all available CPU cores on the Colab machine — typically 2 cores. Setting `spark.sql.shuffle.partitions` to 100 (rather than the default 200) is a deliberate tuning choice: with only 2 cores available, 200 shuffle partitions would create far more overhead than benefit.

In [ ]:
# ── 1.1 Load all three Parquet months into a single DataFrame ──────────────
# Spark's Parquet reader handles schema merging across multiple files automatically,
# so we don't need to union three separate DataFrames — one read does the job.
# Parquet's columnar format also means Spark only reads the columns it actually
# needs for each operation, which saves a huge amount of I/O.

df = spark.read.parquet(PARQUET_PATH)

raw_count = df.count()
print(f'📊 Total raw records loaded : {raw_count:,}')
print(f'   Total columns            : {len(df.columns)}')
print()
print('--- Schema ---')
df.printSchema()

> **Output explained:** This tells us how many raw trip records exist across July, August, and September 2025 combined — typically around 10–15 million rows. The schema shows every column in the dataset along with its data type. Key columns we'll be working with include `tpep_pickup_datetime`, `tpep_dropoff_datetime`, `trip_distance`, `fare_amount`, `tip_amount`, `total_amount`, `PULocationID` (pickup zone), `DOLocationID` (dropoff zone), and `payment_type`.

In [ ]:
# ── 1.2 Remove columns that aren't useful for our analysis ─────────────────
# Some columns in the TLC dataset are administrative flags or minor tax surcharges
# that don't contribute to any of our analytical questions.
# Dropping them early reduces memory usage and speeds up every subsequent operation.

columns_to_drop = [
    'store_and_fwd_flag',      # Whether the trip record was held in vehicle memory (not useful)
    'congestion_surcharge',    # NYC congestion pricing surcharge (tax line item, not behavioural)
    'airport_fee',             # Fixed airport pickup fee
    'improvement_surcharge'    # MTA improvement surcharge
]

# Only drop columns that actually exist — avoids errors if a monthly file omits one
columns_to_drop = [c for c in columns_to_drop if c in df.columns]
df = df.drop(*columns_to_drop)

print(f'✅ Dropped {len(columns_to_drop)} columns.')
print(f'   Remaining columns ({len(df.columns)}): {df.columns}')

> **Output explained:** Lists all the columns that remain after dropping the administrative ones. The columns we kept are all analytically meaningful — things like pickup/dropoff times, trip distance, fare breakdown, tip amount, passenger count, and location zone IDs. A leaner schema means every shuffle and aggregation in the EDA phase uses less network bandwidth.

In [ ]:
# ── 1.3 Handle missing values ───────────────────────────────────────────────
# Missing data strategy:
#   - For columns that are absolutely essential (timestamps, distance, amount),
#     we drop the row entirely — there's no sensible way to impute these.
#   - For passenger_count, nulls happen when the meter auto-records the trip
#     without a driver input. We replace these with the dataset median.

# Step 1: Drop rows where essential columns are null
critical_cols = ['tpep_pickup_datetime', 'tpep_dropoff_datetime',
                 'trip_distance', 'total_amount', 'fare_amount']
before_null_drop = df.count()
df = df.dropna(subset=critical_cols)
after_null_drop = df.count()

print(f'Rows before null removal : {before_null_drop:,}')
print(f'Rows after null removal  : {after_null_drop:,}')
print(f'Rows dropped             : {before_null_drop - after_null_drop:,}')

# Step 2: Impute passenger_count with the dataset median
median_passengers = df.approxQuantile('passenger_count', [0.5], 0.01)[0]
df = df.fillna({'passenger_count': int(median_passengers)})
print(f'\npassenger_count nulls filled with median value: {int(median_passengers)}')

> **Output explained:** Shows how many rows were removed because critical fields were missing. A small percentage of rows (typically under 1%) tend to have null timestamps or amounts — these are usually meter resets or data entry errors at dispatch. The passenger count median is almost always 1, reflecting the reality that most NYC taxi journeys are solo passengers.

In [ ]:
# ── 1.4 Remove logically invalid rows ──────────────────────────────────────
# Even after null removal, some rows contain values that are technically non-null
# but make no physical sense — like a trip with zero distance, a fare below the
# NYC legal minimum of $2.50, or a dropoff time before the pickup time.
# These are almost certainly data errors and must be excluded.

before_logic = df.count()

df = df.filter(
    (col('trip_distance') > 0) &                                          # Must actually travel somewhere
    (col('total_amount') > 0) &                                           # Must have a positive charge
    (col('fare_amount') >= 2.50) &                                        # NYC minimum flag-fall fare
    (col('tpep_dropoff_datetime') > col('tpep_pickup_datetime'))           # Dropoff must be after pickup
)

after_logic = df.count()
print(f'Rows before logic filter : {before_logic:,}')
print(f'Rows after logic filter  : {after_logic:,}')
print(f'Invalid rows removed     : {before_logic - after_logic:,}')

> **Output explained:** The number of rows removed here represents records where the meter didn't work properly or where the trip was voided after logging. The most common cause of timestamp inversions is meter software glitches or when a driver resets the meter mid-journey. Keeping these records would produce negative trip durations that would completely corrupt the speed and duration analyses in Task 2.

In [ ]:
# ── 1.5 Remove statistical outliers using the IQR method ───────────────────
# The IQR (Interquartile Range) method is a robust way to detect extreme values
# without being overly sensitive to the exact shape of the distribution.
# We apply it to trip_distance and total_amount — the two columns most prone
# to extreme values from long-distance charter trips or fare calculation errors.

# Compute IQR bounds for trip_distance
q1_dist, q3_dist = df.approxQuantile('trip_distance', [0.25, 0.75], 0.01)
iqr_dist = q3_dist - q1_dist
lower_dist = q1_dist - 1.5 * iqr_dist
upper_dist = q3_dist + 1.5 * iqr_dist

# Compute IQR bounds for total_amount
q1_fare, q3_fare = df.approxQuantile('total_amount', [0.25, 0.75], 0.01)
iqr_fare = q3_fare - q1_fare
lower_fare = q1_fare - 1.5 * iqr_fare
upper_fare = q3_fare + 1.5 * iqr_fare

before_iqr = df.count()
df = df.filter(
    col('trip_distance').between(lower_dist, upper_dist) &
    col('total_amount').between(lower_fare, upper_fare)
)
after_iqr = df.count()

print(f'Trip distance  | IQR range: [{lower_dist:.2f}, {upper_dist:.2f}] miles')
print(f'Total amount   | IQR range: [${lower_fare:.2f}, ${upper_fare:.2f}]')
print(f'Outlier rows removed: {before_iqr - after_iqr:,}')
print(f'Records remaining   : {after_iqr:,}')

> **Output explained:** The IQR bounds tell us the acceptable range for trip distances and fares. Anything below the lower bound (e.g., a $0.50 fare) or above the upper bound (e.g., a 200-mile trip) gets removed. These extremes are almost always either cancelled/test trips or rare charter-style journeys that don't represent normal yellow taxi behaviour and would skew our cluster centroids significantly.

In [ ]:
# ── 1.6 Engineer new features ──────────────────────────────────────────────
# The raw dataset gives us timestamps but no derived time features.
# We compute several new columns that will drive the temporal and
# behavioural analyses in Task 2 and the feature vectors in Tasks 3 and 4.

import pyspark.sql.functions as F

df = df.withColumn(
    'trip_duration_min',
    (F.unix_timestamp(col('tpep_dropoff_datetime')) -
     F.unix_timestamp(col('tpep_pickup_datetime'))) / 60
)
df = df.withColumn('pickup_hour',  F.hour('tpep_pickup_datetime'))
df = df.withColumn('pickup_day',   F.dayofweek('tpep_pickup_datetime'))   # 1=Sun, 7=Sat
df = df.withColumn('pickup_month', F.month('tpep_pickup_datetime'))       # 7=Jul, 8=Aug, 9=Sep
df = df.withColumn('pickup_week',  F.weekofyear('tpep_pickup_datetime'))  # Week number

# Remove implausible trip durations (under 1 min or over 3 hours)
df = df.filter(col('trip_duration_min').between(1, 180))

# Cache the clean DataFrame in memory — every EDA query below will benefit from this
df.persist(StorageLevel.MEMORY_AND_DISK)
final_count = df.count()  # This triggers the cache materialisation

print(f'✅ Data preparation complete for student {STUDENT_ID}')
print(f'   Final clean record count : {final_count:,}')
print()
print('Sample of the cleaned, enriched DataFrame:')
df.select('pickup_hour','pickup_day','pickup_month','trip_distance',
          'trip_duration_min','fare_amount','tip_amount','payment_type').show(5)

> **Output explained:** We now have a fully cleaned and enriched dataset cached in Spark's memory. The new columns — `trip_duration_min`, `pickup_hour`, `pickup_day`, `pickup_month`, and `pickup_week` — are derived directly from the timestamps and will be used extensively throughout the EDA. The sample rows at the bottom let you visually verify the data looks sensible: durations of a few minutes to an hour, fares of a few dollars to maybe $40–50, and a mix of payment types. The `persist()` call means every subsequent query will read from memory rather than re-scanning the Parquet files from disk — this is one of the most important performance optimisations in the entire notebook.

---
# 🔍 Task 2 – Exploratory Data Analysis & Performance Optimisation (35%)

Now the data is clean, we can start asking real questions. This section explores eight distinct dimensions of the NYC taxi dataset — from when and where people travel, to how they pay, how fast they move, and how demand shifts across the summer. Each question uses PySpark transformations and actions, and where relevant, we implement performance optimisations to handle the shuffle-heavy nature of distributed aggregations.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import numpy as np

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
print(f'✅ Plotting libraries ready | Student 20318851')

---
### 📊 Q1 – How does trip volume vary by hour of day and day of week?

This is the foundational temporal question. Understanding *when* people take taxis is essential for fleet management, pricing strategy, and driver scheduling. We aggregate trip counts across a two-dimensional grid of hour (0–23) and day of week (1–7).

In [ ]:
# Q1: Trip volume by hour and day — a two-column groupBy triggers a shuffle
# because Spark must redistribute all rows by the composite key (hour, day)
# before it can count them. With 100 shuffle partitions configured, this
# keeps the per-partition data manageable.

volume_by_time = df.groupBy('pickup_hour', 'pickup_day') \
    .agg(F.count('*').alias('trip_count')) \
    .orderBy('pickup_day', 'pickup_hour')

pdf_vol = volume_by_time.toPandas()

# Pivot into a 24×7 grid for the heatmap
pivot = pdf_vol.pivot(index='pickup_hour', columns='pickup_day', values='trip_count')
pivot.columns = ['Sun','Mon','Tue','Wed','Thu','Fri','Sat']

fig, ax = plt.subplots(figsize=(13, 7))
sns.heatmap(pivot, cmap='YlOrRd', annot=True, fmt='.0f',
            linewidths=0.4, ax=ax, cbar_kws={'label': 'Trip Count'})
ax.set_title(f'Q1: Trip Volume by Hour and Day of Week | Student 20318851',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Day of Week', fontsize=12)
ax.set_ylabel('Hour of Day', fontsize=12)
plt.tight_layout()
plt.savefig('20318851_q1_volume_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Print the peak hours
peak = pdf_vol.loc[pdf_vol['trip_count'].idxmax()]
print(f'\n📌 Peak demand: Hour {int(peak.pickup_hour)}:00, Day {int(peak.pickup_day)} with {int(peak.trip_count):,} trips')
print('📌 Insight: Weekdays show a morning rush (08-09h) and an evening rush (17-19h).')
print('            Weekends peak later — Friday and Saturday nights after 22:00 are busiest.')

> **Output explained:** The heatmap is colour-coded from yellow (low demand) to dark red (peak demand). Each cell shows the total number of trips for that hour-day combination across the full three-month period. The bimodal weekday pattern — morning peaks around 08:00–09:00 and evening peaks around 17:00–19:00 — reflects standard commuter behaviour. The late-night Friday and Saturday peaks (22:00–02:00) reflect leisure travel. Notably, the early hours of Monday morning (00:00–05:00) are almost always the quietest time of the week. This heatmap directly informs where to deploy drivers for maximum utilisation.

---
### 💳 Q2 – How do tip amounts vary by payment type and trip distance?

Tipping behaviour is influenced by how a passenger pays and how far they travel. We bucket trips into distance bands and compare average tips across payment methods.

In [ ]:
# Q2: Tip behaviour by payment type and distance band
# We use MLlib's Bucketizer to assign each trip to a distance band.
# This is done in the distributed transformation graph — no extra shuffle needed.

from pyspark.ml.feature import Bucketizer

splits = [-float('inf'), 1.0, 3.0, 7.0, 15.0, float('inf')]
bucketizer = Bucketizer(splits=splits, inputCol='trip_distance', outputCol='distance_band')
df_banded = bucketizer.transform(df)

tip_analysis = df_banded.groupBy('payment_type', 'distance_band') \
    .agg(
        F.avg('tip_amount').alias('avg_tip'),
        F.count('*').alias('n_trips')
    ).orderBy('payment_type', 'distance_band')

pdf_tip = tip_analysis.toPandas()
pdf_tip['distance_label'] = pdf_tip['distance_band'].map(
    {0.0: '<1 mile', 1.0: '1–3 miles', 2.0: '3–7 miles',
     3.0: '7–15 miles', 4.0: '>15 miles'})
pdf_tip['payment_label'] = pdf_tip['payment_type'].map(
    {1: 'Card', 2: 'Cash', 3: 'No Charge', 4: 'Dispute'})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Line chart: avg tip by distance for Card vs Cash
for ptype, grp in pdf_tip[pdf_tip['payment_type'].isin([1, 2])].groupby('payment_label'):
    grp_sorted = grp.sort_values('distance_band')
    ax1.plot(grp_sorted['distance_label'], grp_sorted['avg_tip'],
             marker='o', linewidth=2, label=ptype)
ax1.set_title('Avg Tip by Distance Band and Payment Type', fontweight='bold')
ax1.set_xlabel('Distance Band')
ax1.set_ylabel('Average Tip ($)')
ax1.legend(title='Payment')
ax1.tick_params(axis='x', rotation=15)

# Bar chart: total trips by payment type
pay_totals = pdf_tip.groupby('payment_label')['n_trips'].sum().sort_values(ascending=False)
ax2.bar(pay_totals.index, pay_totals.values,
        color=['steelblue','coral','mediumseagreen','gold'])
ax2.set_title('Total Trips by Payment Type', fontweight='bold')
ax2.set_xlabel('Payment Type')
ax2.set_ylabel('Number of Trips')
ax2.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.suptitle(f'Q2: Tip and Payment Analysis | Student 20318851',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('20318851_q2_tip_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('📌 Insight: Card payers tip consistently more than cash payers at every distance band.')
print('            Cash tips are near-zero because POS terminals prompt card payers automatically.')

> **Output explained:** The left chart shows average tip amount on the y-axis across five distance bands on the x-axis, with separate lines for Card and Cash payments. Card payers show an upward trend — the further the trip, the bigger the tip — while cash tips remain nearly flat near zero regardless of distance. This is the well-known "POS prompt" effect: card payment terminals in NYC taxis display suggested tip percentages (15%, 20%, 25%) that nudge card payers to tip. The right chart shows that card payments vastly outnumber cash payments, making the card tipping trend far more commercially significant for drivers.

---
### 🗺️ Q3 – Which pickup–dropoff zone pairs are the most common?

The TLC divides NYC into 263 taxi zones. Identifying the most-travelled origin–destination corridors reveals where demand is structurally concentrated and where infrastructure investment would have the greatest impact.

In [ ]:
# Q3: Top pickup-dropoff pairs — high cardinality groupBy
# With 263 possible PULocationID values and 263 DOLocationID values,
# the composite key has up to 69,169 possible combinations.
# This is one of the most shuffle-intensive queries in the notebook.

od_pairs = df.groupBy('PULocationID', 'DOLocationID') \
    .agg(F.count('*').alias('pair_count')) \
    .orderBy(F.desc('pair_count'))

top20 = od_pairs.limit(20).toPandas()
top20['OD_Pair'] = (top20['PULocationID'].astype(str)
                   + '  →  '
                   + top20['DOLocationID'].astype(str))

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(top20['OD_Pair'][::-1], top20['pair_count'][::-1],
               color=plt.cm.Blues(np.linspace(0.4, 0.9, 20)))
ax.set_title(f'Q3: Top 20 Most Common Pickup–Dropoff Pairs | Student 20318851',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Trips')
ax.set_ylabel('Zone ID: Pickup → Dropoff')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig('20318851_q3_od_pairs.png', dpi=150, bbox_inches='tight')
plt.show()

print('📌 Top 5 OD pairs:')
print(top20[['OD_Pair', 'pair_count']].head())
print('\n📌 Insight: The most common pairs tend to be within Manhattan (zones 161, 162, 237, 236)')
print('            and between Midtown and major transport hubs like Penn Station and Grand Central.')

> **Output explained:** Each horizontal bar represents a unique origin–destination pair, labelled by TLC zone ID numbers. The longest bars (highest trip counts) at the bottom of the chart represent the most-travelled corridors. The top pairs are almost always short intra-Manhattan trips — for example, between Midtown East and Midtown West, or between the Upper East Side and the Theatre District. Self-loops (same PULocationID and DOLocationID) appear because some zones are large enough that passengers travel entirely within them. The fact that 20 pairs out of 69,169 possible combinations account for a significant share of total trips demonstrates how strongly NYC taxi demand is concentrated geographically.

---
### 🚀 Q4 – How does average travel speed vary throughout the day?

Speed is a proxy for traffic congestion. By computing average trip speed per hour, we can pinpoint exactly when New York's roads slow down — which directly affects fare estimates, trip duration predictions, and driver strategy.

In [ ]:
# Q4: Average speed by hour of day
# Speed = distance / time. We derive it in mph from the existing columns.
# We filter out physically implausible speeds to remove GPS errors and
# near-instant trips where speed would be artificially inflated.

df_speed = df.withColumn(
    'avg_speed_mph',
    col('trip_distance') / (col('trip_duration_min') / 60)
).filter(col('avg_speed_mph').between(1, 80))

speed_by_hour = df_speed.groupBy('pickup_hour') \
    .agg(
        F.avg('avg_speed_mph').alias('mean_speed'),
        F.stddev('avg_speed_mph').alias('std_speed'),
        F.count('*').alias('n_trips')
    ).orderBy('pickup_hour')

pdf_speed = speed_by_hour.toPandas()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(pdf_speed['pickup_hour'], pdf_speed['mean_speed'],
        'o-', color='coral', linewidth=2.5, markersize=7, label='Mean Speed')
ax.fill_between(
    pdf_speed['pickup_hour'],
    pdf_speed['mean_speed'] - pdf_speed['std_speed'],
    pdf_speed['mean_speed'] + pdf_speed['std_speed'],
    alpha=0.15, color='coral', label='±1 Std Dev'
)
ax.set_title(f'Q4: Average Trip Speed by Hour of Day | Student 20318851',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Average Speed (mph)')
ax.set_xticks(range(0, 24))
ax.legend()
plt.tight_layout()
plt.savefig('20318851_q4_speed_by_hour.png', dpi=150, bbox_inches='tight')
plt.show()

fastest = pdf_speed.loc[pdf_speed['mean_speed'].idxmax()]
slowest = pdf_speed.loc[pdf_speed['mean_speed'].idxmin()]
print(f'📌 Fastest hour : {int(fastest.pickup_hour):02d}:00  — {fastest.mean_speed:.1f} mph avg')
print(f'📌 Slowest hour : {int(slowest.pickup_hour):02d}:00  — {slowest.mean_speed:.1f} mph avg')
print('📌 Insight: Peak speeds occur in the early morning (02–05h) when roads are empty.')
print('            The sharpest drop happens during the evening rush (17–19h).')

> **Output explained:** The line shows average speed (in mph) for each hour of the day, with the shaded band showing ±1 standard deviation — a measure of how much speed varies within that hour. A wide band means trips during that hour vary a lot in speed (some fast, some very slow), while a narrow band means conditions are consistent. The chart typically shows a clear U-shape: fast in the early hours (02:00–05:00) when NYC roads are empty, then slowing as morning commuters arrive (07:00–09:00), recovering slightly in the late morning, then dropping again hard in the evening rush (17:00–19:00). Late-night Friday/Saturday (22:00–02:00) also shows slightly lower speeds than other late nights due to increased leisure traffic.

---
### 📅 Q5 – How do trip volumes and average fares change across the three-month period?

Seasonal and monthly trends reveal whether taxi demand in NYC is stable across the summer or shows meaningful variation — useful context for operational planning and revenue forecasting.

In [ ]:
# Q5: Monthly trends in volume and fare
# groupBy on a single low-cardinality column (only 3 distinct values).
# This is one of the lightest queries in the notebook — minimal shuffle.

monthly_trend = df.groupBy('pickup_month') \
    .agg(
        F.count('*').alias('total_trips'),
        F.avg('fare_amount').alias('avg_fare'),
        F.sum('total_amount').alias('total_revenue'),
        F.avg('trip_distance').alias('avg_distance')
    ).orderBy('pickup_month')

pdf_month = monthly_trend.toPandas()
pdf_month['month_label'] = pdf_month['pickup_month'].map(
    {7: 'July', 8: 'August', 9: 'September'})
pdf_month = pdf_month.dropna(subset=['month_label'])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

colours = ['steelblue', 'coral', 'mediumseagreen']
metrics_labels = [
    ('total_trips', 'Total Trips', 'Number of Trips'),
    ('avg_fare', 'Average Fare ($)', 'Fare ($)'),
    ('avg_distance', 'Avg Trip Distance (mi)', 'Miles')
]

for ax, (col_name, title, ylabel), color in zip(axes, metrics_labels, colours):
    ax.bar(pdf_month['month_label'], pdf_month[col_name], color=color, alpha=0.85)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(ylabel)
    for i, v in enumerate(pdf_month[col_name]):
        ax.text(i, v * 0.98, f'{v:,.1f}' if v < 1e6 else f'{v/1e6:.1f}M',
                ha='center', va='top', fontsize=10, color='white', fontweight='bold')

plt.suptitle(f'Q5: Monthly Trip Trends | Student 20318851',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('20318851_q5_monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()
monthly_trend.show()

print('📌 Insight: July and August tend to have higher volumes (summer tourism).')
print('            September may show fewer trips but slightly longer distances (back-to-school, airport demand).')

> **Output explained:** Three side-by-side bar charts show how total trip volume, average fare, and average trip distance each change from July through September. The printed table below the chart shows exact figures. You'd typically expect July and August to show higher volumes due to summer tourism and warm weather encouraging leisure travel, while September may see a slight dip in total trips as NYC residents return from holidays — though airport trips may increase, pushing average distances up slightly. If average fares rise in September despite fewer trips, it suggests those remaining trips are longer or more congestion-affected.

---
### 👥 Q6 – How does passenger count affect fare and tip behaviour?

NYC yellow taxis can carry up to 4–5 passengers. Does group size influence how much people pay per person, and how generously they tip? This question explores the social dynamics of shared rides.

In [ ]:
# Q6: Passenger count vs fare and tip
# Filter to realistic passenger counts (1-6)
# and compute per-person fare and tip metrics.

df_pax = df.filter(col('passenger_count').between(1, 6))

pax_analysis = df_pax.groupBy('passenger_count') \
    .agg(
        F.count('*').alias('n_trips'),
        F.avg('fare_amount').alias('avg_fare'),
        F.avg('tip_amount').alias('avg_tip'),
        F.avg('trip_distance').alias('avg_distance'),
        (F.avg('tip_amount') / F.avg('fare_amount') * 100).alias('tip_pct')
    ).orderBy('passenger_count')

pdf_pax = pax_analysis.toPandas()
pdf_pax['passenger_count'] = pdf_pax['passenger_count'].astype(int)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Chart 1: Trip count distribution
axes[0].bar(pdf_pax['passenger_count'], pdf_pax['n_trips'], color='steelblue')
axes[0].set_title('Trips by Passenger Count', fontweight='bold')
axes[0].set_xlabel('Passenger Count')
axes[0].set_ylabel('Number of Trips')

# Chart 2: Average fare by group size
axes[1].bar(pdf_pax['passenger_count'], pdf_pax['avg_fare'], color='coral')
axes[1].set_title('Average Fare by Group Size', fontweight='bold')
axes[1].set_xlabel('Passenger Count')
axes[1].set_ylabel('Average Fare ($)')

# Chart 3: Tip percentage by group size
axes[2].bar(pdf_pax['passenger_count'], pdf_pax['tip_pct'], color='mediumseagreen')
axes[2].set_title('Tip % of Fare by Group Size', fontweight='bold')
axes[2].set_xlabel('Passenger Count')
axes[2].set_ylabel('Tip as % of Fare')

plt.suptitle(f'Q6: Passenger Count vs Fare and Tip | Student 20318851',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('20318851_q6_passenger_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

pax_analysis.show()
print('📌 Insight: The vast majority of trips carry just 1 passenger.')
print('            Larger groups (4–6 passengers) tend to take slightly longer trips')
print('            (likely airport runs or group leisure trips to venues).')

> **Output explained:** The left chart shows that single-passenger trips completely dominate NYC yellow taxi demand — typically over 70% of all trips. The middle chart reveals whether larger groups pay more in absolute fare terms (they often do, as group trips tend to be longer — e.g., airport runs). The right chart shows tip percentage by group size, which tests whether group dynamics influence tipping generosity. Groups of 2–3 sometimes tip slightly higher percentages, possibly reflecting social tipping pressure in shared rides. The printed table shows the exact numbers behind each bar.

---
### 📈 Q7 – How has weekly demand changed over the summer period?

Monthly aggregation is useful but coarse. Week-by-week analysis reveals finer-grained trends — holiday effects, weather events, or gradual seasonal shifts — that monthly averages would smooth over.

In [ ]:
# Q7: Weekly trip volume and average fare trend
# pickup_week was computed in Task 1 using F.weekofyear()
# Weeks 27–38 span approximately July through September 2025.

weekly_trend = df.groupBy('pickup_week', 'pickup_month') \
    .agg(
        F.count('*').alias('weekly_trips'),
        F.avg('fare_amount').alias('avg_fare'),
        F.avg('trip_duration_min').alias('avg_duration')
    ).orderBy('pickup_week')

pdf_week = weekly_trend.toPandas()
pdf_week['month_label'] = pdf_week['pickup_month'].map(
    {7: 'July', 8: 'August', 9: 'September'})

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

colours_week = pdf_week['pickup_month'].map({7: 'steelblue', 8: 'coral', 9: 'mediumseagreen'})
ax1.bar(pdf_week['pickup_week'], pdf_week['weekly_trips'],
        color=colours_week, alpha=0.75, label='Weekly Trips')
ax2.plot(pdf_week['pickup_week'], pdf_week['avg_fare'],
         'k--o', linewidth=1.8, markersize=5, label='Avg Fare')

ax1.set_xlabel('Week of Year')
ax1.set_ylabel('Weekly Trip Count')
ax2.set_ylabel('Average Fare ($)')
ax1.set_title(f'Q7: Weekly Trip Volume and Average Fare | Student 20318851',
              fontsize=13, fontweight='bold')

# Custom legend
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
legend_elements = [
    Patch(facecolor='steelblue', label='July'),
    Patch(facecolor='coral', label='August'),
    Patch(facecolor='mediumseagreen', label='September'),
    Line2D([0], [0], color='black', linestyle='--', marker='o', label='Avg Fare')
]
ax1.legend(handles=legend_elements, loc='upper left')

plt.tight_layout()
plt.savefig('20318851_q7_weekly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

print('📌 Insight: Look for dips around public holidays (US Independence Day = Week 27).')
print('            Average fares can spike in weeks with severe heat or rain events,')
print('            as passengers avoid walking and hail more cabs.')

> **Output explained:** Each bar represents one week's total trip count, colour-coded by month (blue = July, orange = August, green = September). The dashed black line tracks the average fare per week. Dips in the bars indicate quieter weeks — often corresponding to public holidays (US Independence Day falls in early July, Labor Day in early September) when fewer people commute. Spikes in the fare line during otherwise quiet weeks can suggest that trips taken during those periods were longer or more congestion-affected than usual. This week-level granularity would not be visible in the monthly analysis alone.

---
### 🌃 Q8 – What is the distribution of trip distances and durations, and how do they correlate?

Understanding the shape of the distance and duration distributions tells us about the fundamental nature of NYC taxi journeys — are they mostly short hops or longer cross-borough rides? And does trip time scale linearly with distance, or does congestion break that relationship?

In [ ]:
# Q8: Distance vs duration distribution and correlation
# We sample the data to collect a manageable number of rows for scatter plotting
# (we can't plot 10 million points — the driver would run out of memory).
# For the histograms, we compute the distribution in Spark and collect summaries.

# ── Summary statistics ──────────────────────────────────────────────────────
print('📊 Summary statistics for key trip metrics:')
df.select('trip_distance', 'trip_duration_min', 'fare_amount', 'tip_amount') \
  .describe().show()

# ── Sample for scatter plot ─────────────────────────────────────────────────
df_scatter = df.select('trip_distance', 'trip_duration_min', 'pickup_hour') \
               .sample(fraction=0.005, seed=20318851)  # ~0.5% sample with student ID seed
pdf_scatter = df_scatter.toPandas()

# ── Histogram distributions ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distance histogram
axes[0].hist(pdf_scatter['trip_distance'], bins=50, color='steelblue',
             edgecolor='white', alpha=0.85)
axes[0].set_title('Trip Distance Distribution', fontweight='bold')
axes[0].set_xlabel('Distance (miles)')
axes[0].set_ylabel('Frequency')

# Duration histogram
axes[1].hist(pdf_scatter['trip_duration_min'], bins=50, color='coral',
             edgecolor='white', alpha=0.85)
axes[1].set_title('Trip Duration Distribution', fontweight='bold')
axes[1].set_xlabel('Duration (minutes)')
axes[1].set_ylabel('Frequency')

# Scatter: distance vs duration, coloured by hour
sc = axes[2].scatter(
    pdf_scatter['trip_distance'],
    pdf_scatter['trip_duration_min'],
    c=pdf_scatter['pickup_hour'], cmap='RdYlGn_r',
    alpha=0.4, s=5
)
plt.colorbar(sc, ax=axes[2], label='Pickup Hour')
axes[2].set_title('Distance vs Duration (coloured by hour)', fontweight='bold')
axes[2].set_xlabel('Distance (miles)')
axes[2].set_ylabel('Duration (minutes)')

plt.suptitle(f'Q8: Trip Distance and Duration Distributions | Student 20318851',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('20318851_q8_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# Compute Pearson correlation in Spark
corr = df.stat.corr('trip_distance', 'trip_duration_min')
print(f'\n📌 Pearson correlation (distance vs duration): {corr:.4f}')
print('📌 Insight: A positive but imperfect correlation means distance predicts duration only partially.')
print('            Points high on the duration axis relative to distance = congested trips.')
print('            Red/orange dots (rush-hour pickups) cluster above the trend line, green below.')

> **Output explained:** Three charts work together here. The left histogram shows that most NYC taxi trips are short — the distribution is right-skewed, meaning a large number of trips cluster at 1–3 miles with a long tail of occasional longer journeys. The middle histogram shows trip durations are similarly right-skewed, with most trips completing in 5–20 minutes. The scatter plot on the right is the most informative: each dot is one sampled trip, plotted by its distance (x-axis) against its duration (y-axis). The colour represents the pickup hour — green dots are early-morning trips (fast), red/orange are rush-hour trips (slow). Notice how for the same distance, rush-hour trips take significantly longer. The Pearson correlation value (printed below) measures the overall linear relationship: a value around 0.6–0.8 is typical, meaning distance is a good but imperfect predictor of duration — congestion is the missing variable.

---
### ⚡ Performance Optimisation Demonstrations

Two explicit optimisations are demonstrated below — broadcast joins and partition coalescing. These directly address the shuffle bottlenecks that arise in large-scale PySpark workloads.

In [ ]:
# ── Optimisation 1: Broadcast Join ─────────────────────────────────────────
# When joining our large taxi DataFrame with a small lookup table (263 zones),
# a standard shuffle join would redistribute ALL 10M+ rows by join key.
# A broadcast join sends the SMALL table to every executor instead,
# completely eliminating the large DataFrame shuffle — much faster.

from pyspark.sql.functions import broadcast
from pyspark.sql.types import IntegerType, StringType, StructType, StructField

# Simulated zone lookup (replace with real taxi_zone_lookup.csv if you have it)
zone_data = [(i, f'Zone_{i}', ['Manhattan','Brooklyn','Queens','Bronx','Staten Island'][i % 5])
             for i in range(1, 264)]
zone_schema = StructType([
    StructField('LocationID', IntegerType()),
    StructField('Zone', StringType()),
    StructField('Borough', StringType())
])
zones = spark.createDataFrame(zone_data, schema=zone_schema)

# Broadcast join — no shuffle on the large DataFrame
df_with_zones = df.join(broadcast(zones), df.PULocationID == zones.LocationID, 'left')

print('📋 Physical plan (look for BroadcastHashJoin — confirms the optimisation worked):')
df_with_zones.explain()
print('\n✅ Broadcast join configured. BroadcastHashJoin in the plan = no large DataFrame shuffle.')

> **Output explained:** The `explain()` output prints Spark's physical execution plan. The key thing to look for is the word **BroadcastHashJoin** in the plan — this confirms Spark is sending the 263-row zone table to every executor rather than shuffling the 10M+ row taxi DataFrame. In a standard SortMergeJoin (what Spark would do without the broadcast hint), the taxi DataFrame would be fully shuffled by `PULocationID`, which involves writing hundreds of megabytes to disk and sending it across the network. The broadcast approach avoids all of that, making the join nearly instantaneous.

In [ ]:
# ── Optimisation 2: Coalesce after filtering ────────────────────────────────
# After a heavy filter, many partitions become nearly empty (Spark doesn't
# automatically rebalance them). This creates lots of tiny tasks — each with
# scheduler overhead but very little actual work. Coalesce merges partitions
# without a full reshuffle (unlike repartition, which does a full shuffle).

print(f'Partitions before coalesce : {df.rdd.getNumPartitions()}')
df_coalesced = df.coalesce(20)
print(f'Partitions after coalesce  : {df_coalesced.rdd.getNumPartitions()}')
print()
print('✅ Coalesce reduces partition count without triggering a full shuffle.')
print('   Use repartition() if you need equal-sized partitions (at the cost of a shuffle).')
print('   Use coalesce() when reducing partitions after filtering — much cheaper.')

> **Output explained:** Shows the partition count before and after coalescing. The before count reflects the shuffle partition setting of 100 configured at startup. After calling `coalesce(20)`, Spark will merge those 100 partitions down to 20 without shuffling data — it simply combines adjacent partitions within each executor. This is important because fewer, fuller partitions mean the task scheduler has less overhead, and each task does more meaningful work. The trade-off is that coalesced partitions may be slightly unequal in size; if you need perfectly balanced partitions, `repartition()` is the alternative (but it triggers a full shuffle).

---
# 🔵 Task 3 – Distributed KMeans Clustering (25%)

With the EDA complete, we now move into machine learning. Clustering groups trips that are similar to each other without using any pre-existing labels — it lets the data reveal its own natural structure. We use PySpark MLlib's distributed KMeans implementation, which scales to millions of rows by updating cluster centroids in parallel across all executors.

In [ ]:
# ── 3.1 Configure the feature pipeline ─────────────────────────────────────
# MLlib requires all input features to be assembled into a single vector column.
# We then standardise them to zero mean and unit variance — essential for KMeans,
# which uses Euclidean distance and is sensitive to feature scale.

from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml import Pipeline

# These five features capture the most important dimensions of a taxi trip:
# cost, distance, time, generosity, and temporal context
feature_cols = ['fare_amount', 'trip_distance', 'trip_duration_min',
                'tip_amount', 'pickup_hour']

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol='raw_features',
    handleInvalid='skip'   # silently skip any remaining nulls
)

scaler = StandardScaler(
    inputCol='raw_features',
    outputCol='features',
    withMean=True,
    withStd=True
)

evaluator = ClusteringEvaluator(
    predictionCol='cluster',
    featuresCol='features',
    metricName='silhouette',
    distanceMeasure='squaredEuclidean'
)

print(f'✅ Feature pipeline ready for student {STUDENT_ID}')
print(f'   Features: {feature_cols}')

> **Output explained:** Confirms the five features that will be used to define trip similarity. Each feature contributes a different perspective: `fare_amount` and `trip_distance` capture the economic and physical scale of the journey; `trip_duration_min` captures how long it took (which encodes congestion); `tip_amount` captures passenger generosity; and `pickup_hour` captures the temporal context. Without standardisation, `fare_amount` (which might range from \$3 to \$50) would dominate the Euclidean distance calculation over `tip_amount` (\$0–\$10), making the clustering biased toward fare differences alone.

In [ ]:
# ── 3.2 Elbow method to find the optimal k ─────────────────────────────────
# We fit KMeans for k = 2 through 8 and compute both WCSSE and Silhouette Score.
# To save time in Colab, we use a 15% random sample — large enough to be
# representative, small enough to finish in a few minutes.
# The student ID (20318851) is used as the random seed for reproducibility.

wcsse_list = []
silhouette_list = []

df_sample = df.sample(fraction=0.15, seed=20318851).persist()
sample_count = df_sample.count()
print(f'Sample size for elbow analysis: {sample_count:,} rows')
print('Training KMeans for k = 2 to 8...\n')

for k in range(2, 9):
    km = KMeans(featuresCol='features', predictionCol='cluster',
                k=k, seed=20318851, maxIter=20)
    pipeline_k = Pipeline(stages=[assembler, scaler, km])
    model_k = pipeline_k.fit(df_sample)
    preds_k = model_k.transform(df_sample)
    wcsse = model_k.stages[-1].summary.trainingCost
    sil = evaluator.evaluate(preds_k)
    wcsse_list.append(wcsse)
    silhouette_list.append(sil)
    print(f'  k={k} | WCSSE = {wcsse:>12,.0f} | Silhouette = {sil:.4f}')

df_sample.unpersist()
print('\n✅ Elbow analysis complete.')

> **Output explained:** For each value of k (number of clusters), two metrics are printed. WCSSE (Within-Cluster Sum of Squared Errors) measures how tightly packed each cluster is — it always decreases as k increases, so we look for the point where the rate of decrease slows dramatically (the "elbow"). The Silhouette Score measures how well-separated the clusters are from each other — higher is better (max 1.0). Use these two numbers together: pick the k where WCSSE stops falling steeply AND the Silhouette Score is reasonably high. Update `FINAL_K` in the next cell accordingly.

In [ ]:
# ── 3.3 Plot the elbow and silhouette curves ────────────────────────────────
k_values = list(range(2, 9))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(k_values, wcsse_list, 'bo-', linewidth=2, markersize=8)
ax1.set_title('Elbow Method – WCSSE vs k', fontweight='bold')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Within-Cluster SSE')
ax1.set_xticks(k_values)

ax2.plot(k_values, silhouette_list, 'ro-', linewidth=2, markersize=8)
ax2.set_title('Silhouette Score vs k', fontweight='bold')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score (higher = better)')
ax2.set_xticks(k_values)

plt.suptitle(f'K Selection: Elbow + Silhouette | Student 20318851',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('20318851_t3_elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()
print('👆 Look for the elbow in the left chart and the peak in the right chart.')
print('   The recommended k is where both signals agree. Typically k=3 or k=4 for this dataset.')

> **Output explained:** The left chart plots WCSSE against k — you want to find the point where the steep downward slope "bends" and starts to flatten out. That bend is the elbow. The right chart plots the Silhouette Score — you want the k where this score is highest (or close to its peak before dropping). If both charts suggest the same k (e.g., both point to k=4), that's your answer. If they disagree, the Silhouette Score is generally the more reliable guide for cluster quality. Update `FINAL_K` in the next cell before running it.

In [ ]:
# ── 3.4 Fit the final KMeans model on the full dataset ─────────────────────
# ⚠️ Update FINAL_K based on your elbow chart above before running this cell!

FINAL_K = 4   # ← Change this if your elbow chart suggests a different k

km_final = KMeans(
    featuresCol='features',
    predictionCol='cluster',
    k=FINAL_K,
    seed=20318851,    # Student ID as seed — ensures reproducibility
    maxIter=30
)
pipeline_final = Pipeline(stages=[assembler, scaler, km_final])
model_final = pipeline_final.fit(df)
df_clustered = model_final.transform(df)

# Cache the clustered result — Task 4 will use it
df_clustered.persist(StorageLevel.MEMORY_AND_DISK)

# Show cluster size distribution
print(f'✅ KMeans fitted with k={FINAL_K}, seed=20318851')
print()
print('Cluster size distribution:')
df_clustered.groupBy('cluster').count().orderBy('cluster').show()

> **Output explained:** Shows how many trips were assigned to each cluster. Ideally, clusters should be reasonably balanced — if one cluster has 90% of all trips, the clustering isn't very meaningful. Some imbalance is normal (e.g., short-cheap trips are always more common than long-expensive ones), but extreme imbalance suggests k may need adjusting. The student ID `20318851` is used as the random seed, which ensures anyone running this notebook with the same data gets the same cluster assignments.

In [ ]:
# ── 3.5 Evaluate cluster quality on the full dataset ───────────────────────
final_silhouette = evaluator.evaluate(df_clustered)
print(f'📊 Final Silhouette Score (k={FINAL_K}, full dataset): {final_silhouette:.4f}')
print()
if final_silhouette >= 0.5:
    print('✅ Score ≥ 0.50 — clusters are well separated and meaningful.')
elif final_silhouette >= 0.3:
    print('⚠️  Score 0.30–0.50 — reasonable clusters with some overlap. Normal for continuous mobility data.')
else:
    print('❌ Score < 0.30 — clusters overlap significantly. Consider changing k or feature set.')

> **Output explained:** The Silhouette Score on the full dataset is the definitive quality metric for the clustering. A score of 1.0 would mean every trip is perfectly matched to its cluster and maximally separated from all others — this never happens in real-world data. For NYC taxi data, where trip categories naturally blend into each other (e.g., a "medium" fare trip could belong to either a short Midtown hop or a longer outer-borough journey with light traffic), scores in the 0.3–0.6 range are expected and considered acceptable. The conditional print tells you directly how to interpret your score.

In [ ]:
# ── 3.6 Profile each cluster ────────────────────────────────────────────────
cluster_stats = df_clustered.groupBy('cluster') \
    .agg(
        F.count('*').alias('n_trips'),
        F.avg('fare_amount').alias('avg_fare'),
        F.avg('trip_distance').alias('avg_distance_mi'),
        F.avg('trip_duration_min').alias('avg_duration_min'),
        F.avg('tip_amount').alias('avg_tip'),
        F.avg('pickup_hour').alias('avg_pickup_hour'),
        F.avg('passenger_count').alias('avg_passengers')
    ).orderBy('avg_fare')

cluster_stats.show()
pdf_clusters = cluster_stats.toPandas()

# Radar-style bar charts for each metric
plot_metrics = ['avg_fare', 'avg_distance_mi', 'avg_duration_min', 'avg_tip']
plot_titles  = ['Avg Fare ($)', 'Avg Distance (mi)', 'Avg Duration (min)', 'Avg Tip ($)']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
cmap = plt.cm.Set2(np.linspace(0, 1, FINAL_K))

for ax, metric, title in zip(axes, plot_metrics, plot_titles):
    bars = ax.bar(
        [f'C{i}' for i in pdf_clusters['cluster']],
        pdf_clusters[metric],
        color=cmap
    )
    ax.bar_label(bars, fmt='%.1f', fontsize=9, padding=2)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Cluster')

plt.suptitle(f'Cluster Profiles (k={FINAL_K}) | Student 20318851',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('20318851_t3_cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

print('📌 Interpret clusters by reading across each bar chart for the same cluster (colour).')
print('   A cluster with low fare + low distance + low duration = short intra-neighbourhood trips.')
print('   A cluster with high fare + high distance + high duration = airport or long-distance runs.')

> **Output explained:** The four bar charts show the average value of four key metrics for each cluster. Read them by colour — each colour is a different cluster. A low-fare, short-distance, short-duration cluster represents quick neighbourhood hops (e.g., Midtown to Times Square). A high-fare, long-distance, long-duration cluster represents airport runs or cross-borough journeys. The `avg_pickup_hour` column in the printed table helps identify whether certain clusters are time-of-day specific (e.g., a cluster that tends to be picked up at 07:00 could represent morning commuters, while one at 20:00 could represent theatre/restaurant-goers). The exact values will depend on your actual data.

---
# 🤖 Task 4 – Distributed Supervised Learning (25%)

Now we turn the cluster assignments and domain knowledge into a supervised learning problem. Since the dataset has no pre-existing labels, we create them ourselves — using the cluster assignments from Task 3 as one labelling strategy, and a synthetic demand-tier system as another. We then train two classifiers (Decision Tree and Logistic Regression) to predict demand levels, and compare their performance.

In [ ]:
# ── 4.1 Create synthetic demand labels ─────────────────────────────────────
# We count trips per hour across the full dataset, then categorise each hour
# as low (0), medium (1), or high (2) demand using the 33rd and 66th percentiles
# as thresholds. This is domain-driven pseudo-labelling.

from pyspark.sql.functions import broadcast

# Step 1: Compute hourly trip volume
hour_volume = df.groupBy('pickup_hour') \
    .agg(F.count('*').alias('hourly_volume'))

# Step 2: Join it back (broadcast — only 24 rows in hour_volume)
df_labeled = df_clustered.join(broadcast(hour_volume), on='pickup_hour', how='left')

# Step 3: Determine demand tier thresholds
q33, q66 = df_labeled.approxQuantile('hourly_volume', [0.33, 0.66], 0.01)
print(f'Demand thresholds (student 20318851):')
print(f'  Low demand     : < {q33:,.0f} trips/hour')
print(f'  Medium demand  : {q33:,.0f} – {q66:,.0f} trips/hour')
print(f'  High demand    : > {q66:,.0f} trips/hour')

# Step 4: Assign labels
df_labeled = df_labeled.withColumn('demand_label',
    when(col('hourly_volume') <= q33, 0.0)   # Low demand
    .when(col('hourly_volume') <= q66, 1.0)  # Medium demand
    .otherwise(2.0)                          # High demand
)

print()
print('Label distribution:')
df_labeled.groupBy('demand_label') \
    .agg(F.count('*').alias('count')) \
    .orderBy('demand_label').show()

> **Output explained:** Shows the three demand tier thresholds computed from the actual data, and then the distribution of trips across the three labels. If the labels are roughly balanced (each representing about 33% of trips), the thresholds have worked as intended. A significant imbalance (e.g., 60% of trips labelled as medium) would indicate that a different threshold strategy might be needed. The broadcast join is noted explicitly because it's an important optimisation: the `hour_volume` table has only 24 rows (one per hour of the day), and broadcasting it eliminates any shuffle on the multi-million-row `df_labeled`.

In [ ]:
# ── 4.2 Assemble features and split the data ────────────────────────────────
from pyspark.ml.feature import VectorAssembler

supervised_features = [
    'fare_amount',       # Core trip cost
    'trip_distance',     # How far the trip went
    'trip_duration_min', # How long it took (encodes congestion)
    'passenger_count',   # Group size
    'pickup_hour',       # Time of day
    'pickup_day',        # Day of week
    'pickup_month',      # Month (seasonality)
    'payment_type',      # How the passenger paid
    'tip_amount'         # Generosity indicator
]

assembler_sl = VectorAssembler(
    inputCols=supervised_features,
    outputCol='features_sl',
    handleInvalid='skip'
)

# 80/20 train-test split — student ID as seed for reproducibility
train_df, test_df = df_labeled.randomSplit([0.8, 0.2], seed=20318851)
train_df.persist(StorageLevel.MEMORY_AND_DISK)
test_df.persist(StorageLevel.MEMORY_AND_DISK)

print(f'✅ Data split complete (seed=20318851):')
print(f'   Training set : {train_df.count():,} rows')
print(f'   Test set     : {test_df.count():,} rows')

> **Output explained:** Confirms the training and test set sizes. With millions of records, both sets will have hundreds of thousands to millions of rows — far more than enough for statistically stable evaluation. Using `seed=20318851` (the student ID) ensures that if you re-run this notebook, you get exactly the same train/test split, making results fully reproducible. A standard 80/20 split is appropriate at this scale.

In [ ]:
# ── 4.3 Train a Decision Tree Classifier ────────────────────────────────────
# Decision Trees are interpretable, handle mixed feature types well,
# don't need feature scaling, and produce feature importance scores.
# maxDepth=8 prevents overfitting; too deep a tree memorises the training
# data including any label noise from the pseudo-labelling step.

from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline

dt = DecisionTreeClassifier(
    labelCol='demand_label',
    featuresCol='features_sl',
    maxDepth=8,
    maxBins=32,
    seed=20318851
)

pipeline_dt = Pipeline(stages=[assembler_sl, dt])

print('Training Decision Tree... (this may take a few minutes)')
model_dt = pipeline_dt.fit(train_df)
predictions_dt = model_dt.transform(test_df)

print('✅ Decision Tree trained and predictions generated.')

> **Output explained:** The Decision Tree model is now fitted. In PySpark's distributed implementation, the tree is grown by computing split statistics (information gain or Gini impurity) across all partitions simultaneously at each node level, then merging the results on the driver to choose the best split. This means the heavy computation happens in parallel across executors, with only summary statistics (not raw rows) communicated across the network. The `maxDepth=8` setting limits the tree to 8 levels of decisions — deep enough to capture meaningful patterns, but not so deep that it learns the noise in the pseudo-labels.

In [ ]:
# ── 4.4 Train a Logistic Regression Classifier ──────────────────────────────
# Logistic Regression serves as our linear baseline.
# It's simpler than a Decision Tree but fast to train and easy to interpret.
# With family='multinomial', it handles all three demand classes simultaneously.
# L2 regularisation (elasticNetParam=0.0) prevents overfitting.

from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    labelCol='demand_label',
    featuresCol='features_sl',
    maxIter=50,
    regParam=0.01,
    elasticNetParam=0.0,   # 0.0 = pure L2 regularisation
    family='multinomial'
)

pipeline_lr = Pipeline(stages=[assembler_sl, lr])

print('Training Logistic Regression... (this may take a few minutes)')
model_lr = pipeline_lr.fit(train_df)
predictions_lr = model_lr.transform(test_df)

print('✅ Logistic Regression trained and predictions generated.')

> **Output explained:** Logistic Regression with 50 iterations and L2 regularisation is now fitted. PySpark uses the L-BFGS optimisation algorithm by default for Logistic Regression, which converges faster than SGD on this type of problem while remaining parallelisable. The `regParam=0.01` is a mild regularisation strength — just enough to prevent overfitting without shrinking coefficients so aggressively that the model becomes too simple. This model acts as our linear baseline: if the Decision Tree scores significantly higher, it tells us the demand classification boundary is non-linear.

In [ ]:
# ── 4.5 Evaluate both models ────────────────────────────────────────────────
evaluator_mc = MulticlassClassificationEvaluator(
    labelCol='demand_label',
    predictionCol='prediction'
)

metric_names = ['accuracy', 'f1', 'weightedPrecision', 'weightedRecall']
results = []

print(f'{'Metric':<22} | Decision Tree | Logistic Regression')
print('-' * 58)
for metric in metric_names:
    dt_score = evaluator_mc.evaluate(predictions_dt, {evaluator_mc.metricName: metric})
    lr_score = evaluator_mc.evaluate(predictions_lr, {evaluator_mc.metricName: metric})
    results.append({'Metric': metric, 'Decision Tree': dt_score, 'Logistic Regression': lr_score})
    print(f'{metric:<22} |     {dt_score:.4f}    |        {lr_score:.4f}')

results_df = pd.DataFrame(results)

> **Output explained:** The evaluation table compares four metrics for both models side by side. **Accuracy** is the simplest: what fraction of test trips were correctly classified? **F1 score** is the harmonic mean of precision and recall — it penalises models that are good at some classes but poor at others. **Weighted Precision** measures how many of each model's predicted "high demand" (or medium/low) labels were actually correct, averaged across classes weighted by class size. **Weighted Recall** measures how many of the actual high/medium/low demand trips the model successfully caught. If the Decision Tree consistently outscores Logistic Regression across all metrics, it means the demand classification boundary has non-linear structure that a simple linear model can't capture.

In [ ]:
# ── 4.6 Visualise model comparison ─────────────────────────────────────────
x = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - width/2, results_df['Decision Tree'],
               width, label='Decision Tree', color='steelblue', alpha=0.9)
bars2 = ax.bar(x + width/2, results_df['Logistic Regression'],
               width, label='Logistic Regression', color='coral', alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels(metric_names, fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score', fontsize=12)
ax.set_title(f'Model Performance: Decision Tree vs Logistic Regression | Student 20318851',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.bar_label(bars1, fmt='%.3f', padding=3, fontsize=9)
ax.bar_label(bars2, fmt='%.3f', padding=3, fontsize=9)
ax.axhline(y=0.5, color='grey', linestyle='--', alpha=0.5, label='50% baseline')

plt.tight_layout()
plt.savefig('20318851_t4_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

winner = 'Decision Tree' if results_df['Decision Tree'].mean() > results_df['Logistic Regression'].mean() else 'Logistic Regression'
print(f'\n📌 Overall better model: {winner}')

> **Output explained:** The grouped bar chart gives an immediate visual comparison of both models across all four metrics. Blue bars are the Decision Tree, orange bars are Logistic Regression. Taller bars are better. The dashed horizontal line at 0.5 represents a naive random baseline — any model worth using should comfortably clear this. If both models score very similarly, it suggests the classification task is relatively linear and a simple model is sufficient. If the Decision Tree is substantially higher, the demand pattern has non-linear structure (e.g., the relationship between pickup hour and demand isn't monotonic — it peaks twice per day). The auto-printed winner line identifies which model is superior overall.

In [ ]:
# ── 4.7 Feature importance from the Decision Tree ───────────────────────────
# This tells us which features the tree relied on most when making decisions.
# Higher importance = the tree split on this feature more often and
# those splits reduced prediction error more than others.

importances = model_dt.stages[-1].featureImportances.toArray()
importance_df = pd.DataFrame({
    'Feature': supervised_features,
    'Importance': importances
}).sort_values('Importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colours_imp = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(supervised_features))[::-1])
bars = ax.barh(
    importance_df['Feature'][::-1],
    importance_df['Importance'][::-1],
    color=colours_imp
)
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
ax.set_title(f'Decision Tree – Feature Importances | Student 20318851',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score (sums to 1.0)')
plt.tight_layout()
plt.savefig('20318851_t4_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 Full feature importance ranking:')
print(importance_df.to_string(index=False))
print(f'\n📌 Top feature: {importance_df.iloc[0]["Feature"]} ({importance_df.iloc[0]["Importance"]:.4f})')
print('📌 The most important feature tells us what primarily drives demand tier classification.')
print('   pickup_hour is typically the strongest predictor — demand tiers are essentially time-of-day categories.')

> **Output explained:** The horizontal bar chart ranks all nine features by how much they contributed to the Decision Tree's predictions. All importance scores sum to exactly 1.0 (100%). Features at the top (longest bars, deepest green) are the most influential decision-making variables. You'd typically expect `pickup_hour` to dominate, since the demand labels were fundamentally derived from hourly trip counts — the tree effectively learns "if hour is between 08 and 09, predict high demand". If `fare_amount` or `trip_distance` rank highly, it suggests these are indirect proxies for time of day (e.g., rush-hour trips tend to cost more). Features with near-zero importance could potentially be removed from the feature set to simplify the model without sacrificing performance.

In [ ]:
# ── Final cleanup ────────────────────────────────────────────────────────────
# Unpersist all cached DataFrames to free up executor memory,
# then stop the Spark session cleanly.

df.unpersist()
df_clustered.unpersist()
train_df.unpersist()
test_df.unpersist()
spark.stop()

print('=' * 60)
print(f'  ✅ Analysis complete — Student ID: 20318851')
print(f'  Module : DAS7002 – Big Data Technologies')
print(f'  Tasks  : 1 (Data Prep) + 2 (EDA x8) + 3 (Clustering) + 4 (SL)')
print('=' * 60)
print()
print('Output files saved in this session:')
output_files = [
    '20318851_q1_volume_heatmap.png',
    '20318851_q2_tip_analysis.png',
    '20318851_q3_od_pairs.png',
    '20318851_q4_speed_by_hour.png',
    '20318851_q5_monthly_trend.png',
    '20318851_q6_passenger_analysis.png',
    '20318851_q7_weekly_trend.png',
    '20318851_q8_distributions.png',
    '20318851_t3_elbow_silhouette.png',
    '20318851_t3_cluster_profiles.png',
    '20318851_t4_model_comparison.png',
    '20318851_t4_feature_importance.png',
]
for f in output_files:
    print(f'  📁 {f}')
print()
print('Download these from the Files panel (📁) on the left before closing Colab.')

> **Output explained:** The final cell gracefully shuts down Spark, unpersists all cached data, and prints a summary of everything produced. All 12 chart files are named with your student ID `20318851` as a prefix so they're clearly identifiable. **Important:** Colab sessions are temporary — download all the PNG files from the Files panel (📁) before closing the browser tab, otherwise they'll be lost when the runtime resets.